# 🔥 Tarification du risque incendie – Prime pure d'assurance

**Data challenge** : prédire la **charge sinistre incendie** de contrats d'assurance professionnels (multirisque entreprise).

La charge d'un contrat se décompose selon l'approche actuarielle classique **fréquence × coût moyen** :

$$\text{CHARGE} = \underbrace{\text{FREQ}}_{\text{nb sinistres / an}} \times \underbrace{\text{CM}}_{\text{coût moyen d'un sinistre}} \times \underbrace{\text{ANNEE\_ASSURANCE}}_{\text{exposition}}$$

**Métrique d'évaluation** : RMSE sur `CHARGE`.

### Plan
1. Chargement des données
2. Exploration des cibles
3. Préparation des variables
4. Jeu de validation commun et modèle de référence
5. Modèles : GLM · HistGradientBoosting · LightGBM · XGBoost
6. Comparaison des modèles
7. Modèle final et fichier de soumission

## 0. Imports et configuration

In [ ]:
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import PoissonRegressor, GammaRegressor
from sklearn.ensemble import HistGradientBoostingRegressor

import lightgbm as lgb
import xgboost as xgb

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)

SEED = 42

# Le notebook fonctionne qu'il soit lancé depuis la racine du projet ou depuis notebooks/
RACINE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = RACINE / "data"
SORTIES = RACINE / "outputs"
SORTIES.mkdir(exist_ok=True)

## 1. Chargement des données

| Archive | Fichier | Contenu |
|---|---|---|
| `Dataset2.zip` | `train_input_*.csv` | 383 610 contrats × 374 colonnes (variables explicatives) |
| `Dataset1.zip` | `train_output_*.csv` | Cibles : `FREQ`, `CM`, `ANNEE_ASSURANCE`, `CHARGE` |
| `Dataset1.zip` | `test_input_*.csv` | 95 852 contrats à prédire |
| – | `dictionnaire_variables.xlsx` | Libellé de chaque variable |

Les CSV sont lus **directement dans les archives**, sans décompression préalable.

Lues telles quelles, les colonnes texte occupent **plus de 5 Go** en mémoire. On les convertit en type `category`, ce qui divise l'empreinte mémoire par 5 à 10.

In [ ]:
import zipfile


def charger(archive, prefixe):
    # Lit dans l'archive le CSV dont le nom commence par `prefixe`,
    # puis convertit les colonnes texte en 'category'.
    with zipfile.ZipFile(DATA / archive) as z:
        nom = next(n for n in z.namelist() if Path(n).name.startswith(prefixe) and n.endswith(".csv"))
        with z.open(nom) as f:
            df = pd.read_csv(f, low_memory=False)
    texte = df.select_dtypes(include=["object", "string"]).columns
    df[texte] = df[texte].astype("category")
    return df

t0 = time.time()
X_brut = charger("Dataset2.zip", "train_input")
y = charger("Dataset1.zip", "train_output")
X_test_brut = charger("Dataset1.zip", "test_input")

assert (X_brut["ID"].values == y["ID"].values).all(), "Les ID de X et y ne sont pas alignés"

print(f"Chargement en {time.time() - t0:.0f} s")
for nom, df in [("X_train", X_brut), ("y_train", y), ("X_test", X_test_brut)]:
    print(f"{nom:8s}: {df.shape[0]:>7,} lignes × {df.shape[1]:>3} colonnes "
          f"— {df.memory_usage(deep=True).sum() / 1e6:,.0f} Mo")

## 2. Exploration des cibles

In [ ]:
# Nombre de sinistres = fréquence × exposition (utile pour les statistiques et les pondérations)
y["NB_SINISTRES"] = (y["FREQ"] * y["ANNEE_ASSURANCE"]).round()

resume = pd.Series({
    "Contrats": len(y),
    "Exposition totale (années)": y["ANNEE_ASSURANCE"].sum(),
    "Contrats sinistrés": (y["FREQ"] > 0).sum(),
    "Part de contrats sinistrés (%)": 100 * (y["FREQ"] > 0).mean(),
    "Fréquence (sinistres pour 1 000 années)": 1000 * y["NB_SINISTRES"].sum() / y["ANNEE_ASSURANCE"].sum(),
    "Coût moyen d'un sinistre": y["CHARGE"].sum() / y["NB_SINISTRES"].sum(),
    "Prime pure moyenne (charge / an)": y["CHARGE"].sum() / y["ANNEE_ASSURANCE"].sum(),
    "Charge maximale": y["CHARGE"].max(),
    "Coûts moyens négatifs (recours)": (y["CM"] < 0).sum(),
})
resume.round(1).to_frame("valeur")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

y["ANNEE_ASSURANCE"].plot.hist(bins=50, ax=axes[0], color="#4C72B0")
axes[0].set_title("Exposition (années d'assurance)")
axes[0].set_ylabel("Nombre de contrats")

y.loc[y["NB_SINISTRES"] > 0, "NB_SINISTRES"].value_counts().sort_index().plot.bar(ax=axes[1], color="#4C72B0")
axes[1].set_title("Nombre de sinistres (contrats sinistrés)")

charges = y.loc[y["CHARGE"] > 0, "CHARGE"]
axes[2].hist(np.log10(charges), bins=50, color="#DD8452")
axes[2].set_title("Charge des contrats sinistrés (log10)")
axes[2].set_xlabel("log10(CHARGE)")

plt.tight_layout()
plt.show()

tri = charges.sort_values(ascending=False)
for part in (0.01, 0.10):
    poids = tri.head(int(len(tri) * part)).sum() / tri.sum()
    print(f"Les {part:.0%} de contrats sinistrés les plus coûteux représentent {poids:.0%} de la charge totale.")

**À retenir**

- Moins de **1 % des contrats** ont un sinistre : la cible `CHARGE` est nulle dans plus de 99 % des cas.
- La charge est **très asymétrique** : quelques sinistres graves concentrent une grande part du coût total. Le RMSE est donc dominé par ces gros sinistres, ce qui le rend **bruité** d'un échantillon à l'autre.
- Quelques coûts moyens sont **négatifs** (probablement des recours). Ils sont exclus de l'apprentissage du coût moyen, qui doit rester positif.
- `ANNEE_ASSURANCE` est une **exposition** : elle n'entre pas comme variable explicative. Elle sert de poids pour la fréquence et de multiplicateur final.

## 3. Préparation des variables

- **Retrait** de `ID` et `ANNEE_ASSURANCE`, ainsi que des colonnes constantes.
- **Variables catégorielles** (≈ 280) : *count encoding*, c'est-à-dire que chaque modalité est remplacée par son effectif dans le jeu d'entraînement. Les valeurs manquantes forment une modalité à part, et une modalité absente du train vaut 0.
- **Variables numériques** : les valeurs manquantes sont conservées, car les modèles d'arbres les gèrent nativement. Le GLM les impute par la médiane dans son propre pipeline.

In [ ]:
exposition = y["ANNEE_ASSURANCE"].to_numpy()
exposition_test = X_test_brut["ANNEE_ASSURANCE"].to_numpy()
ids_test = X_test_brut["ID"].to_numpy()

features = [c for c in X_brut.columns if c not in ("ID", "ANNEE_ASSURANCE")]
constantes = [c for c in features if X_brut[c].nunique(dropna=False) <= 1]
features = [c for c in features if c not in constantes]

cat_cols = [c for c in features if isinstance(X_brut[c].dtype, pd.CategoricalDtype)]
num_cols = [c for c in features if c not in cat_cols]


def count_encoding(train, autres, colonnes):
    # Remplace chaque modalité par son effectif dans `train` (NaN = modalité à part).
    train, autres = train.copy(), [df.copy() for df in autres]
    for c in colonnes:
        effectifs = train[c].astype(str).value_counts()
        for df in [train, *autres]:
            df[c] = df[c].astype(str).map(effectifs).fillna(0).astype("float32")
    return train, autres


X_all, (X_test,) = count_encoding(X_brut[features], [X_test_brut[features]], cat_cols)
X_all = X_all.astype("float32")
X_test = X_test.astype("float32")
del X_brut, X_test_brut

print(f"Colonnes constantes retirées : {len(constantes)}")
print(f"Variables retenues           : {len(features)} ({len(cat_cols)} catégorielles, {len(num_cols)} numériques)")
print(f"X_all  : {X_all.shape}  —  X_test : {X_test.shape}")

## 4. Jeu de validation commun et modèle de référence

Tous les modèles sont évalués sur **le même jeu de validation** (20 %, stratifié sur la présence d'un sinistre), ce qui rend leurs RMSE directement comparables.

Deux indicateurs sont suivis :
- **RMSE** sur `CHARGE`, la métrique du challenge ;
- **Gini normalisé** : il mesure le **pouvoir de segmentation**, c'est-à-dire la capacité à classer les contrats du moins risqué au plus risqué. Il vaut 0 pour un classement aléatoire et 1 pour un classement parfait. C'est l'indicateur de référence en tarification, et il est beaucoup moins sensible aux sinistres extrêmes que le RMSE.
- **Ratio charge prédite / charge observée** sur la validation, qui mesure si le modèle tarifie le bon niveau global (idéal : 1,00).

Le **modèle de référence** applique à chaque contrat la prime pure moyenne multipliée par son exposition. Un modèle utile doit faire mieux que lui.

In [ ]:
sinistre = (y["FREQ"] > 0).to_numpy()
idx_tr, idx_val = train_test_split(
    np.arange(len(y)), test_size=0.2, random_state=SEED, stratify=sinistre
)

X_tr, X_val = X_all.iloc[idx_tr], X_all.iloc[idx_val]
y_tr, y_val = y.iloc[idx_tr], y.iloc[idx_val]
expo_tr, expo_val = exposition[idx_tr], exposition[idx_val]
charge_val = y_val["CHARGE"].to_numpy()

# Coût moyen : uniquement les contrats sinistrés avec un coût positif,
# pondérés par leur nombre de sinistres
sev_tr = (y_tr["FREQ"] > 0) & (y_tr["CM"] > 0)
sev_val = (y_val["FREQ"] > 0) & (y_val["CM"] > 0)

print(f"Entraînement : {len(idx_tr):,} contrats ({sev_tr.sum():,} sinistrés)")
print(f"Validation   : {len(idx_val):,} contrats ({sev_val.sum():,} sinistrés)")

resultats = {}


def gini_normalise(y_obs, y_pred):
    # Gini de la courbe de concentration de y_obs triée par y_pred décroissant, rapporté au Gini parfait.
    def gini(score):
        cumul = np.cumsum(y_obs[np.argsort(-score, kind="stable")]) / y_obs.sum()
        return cumul.mean() - (len(y_obs) + 1) / (2 * len(y_obs))
    return gini(np.asarray(y_pred)) / gini(np.asarray(y_obs))


def evaluer(nom, charge_pred, duree=None):
    rmse = np.sqrt(mean_squared_error(charge_val, charge_pred))
    gini = gini_normalise(charge_val, charge_pred)
    ratio = charge_pred.sum() / charge_val.sum()
    resultats[nom] = {"RMSE validation": rmse, "Gini normalisé": gini,
                      "Charge prédite / observée": ratio, "Durée (s)": duree}
    print(f"{nom:<38s} RMSE = {rmse:,.1f}   Gini = {gini:.3f}   ratio = {ratio:.3f}")


prime_pure_moyenne = y_tr["CHARGE"].sum() / expo_tr.sum()
evaluer("Référence : prime pure moyenne", prime_pure_moyenne * expo_val)

## 5. Modèles

Chaque modèle (sauf XGBoost) suit l'approche **fréquence × coût moyen** :

| Composante | Données | Loi / objectif | Poids |
|---|---|---|---|
| Fréquence `FREQ` | tous les contrats | Poisson | exposition |
| Coût moyen `CM` | contrats sinistrés, `CM > 0` | Gamma | nombre de sinistres |

La charge prédite vaut ensuite `FREQ × CM × exposition`.

**Équilibrage** : avec un lien log, les modèles Gamma reproduisent mal la **moyenne** d'une distribution à queue lourde, et peuvent sous-estimer le coût moyen. On applique donc un coefficient d'équilibrage, calculé sur le train, pour que la **charge totale prédite égale la charge observée**. C'est une étape standard en tarification : le tarif doit couvrir la sinistralité globale.

In [ ]:
def poids_sev(df):
    return df["NB_SINISTRES"].clip(lower=1).to_numpy()


def coef_equilibrage(modele_freq, modele_cm, X, charge_obs, expo):
    # Coefficient tel que la charge totale prédite = la charge totale observée.
    charge_pred = modele_freq.predict(X) * modele_cm.predict(X) * expo
    return np.clip(charge_obs, 0, None).sum() / charge_pred.sum()


equilibrage = {}


def evaluer_freq_cm(nom, modele_freq, modele_cm, fit_freq=None, fit_cm=None,
                    poids_freq="sample_weight", poids_cm="sample_weight"):
    # Entraîne fréquence + coût moyen sur le train, puis évalue la charge sur la validation.
    # `poids_*` : nom de l'argument de pondération (différent pour un Pipeline scikit-learn).
    t0 = time.time()
    X_sin, cm_obs, nb_sin = X_tr[sev_tr.values], y_tr.loc[sev_tr, "CM"], poids_sev(y_tr[sev_tr])
    modele_freq.fit(X_tr, y_tr["FREQ"], **{poids_freq: expo_tr}, **(fit_freq or {}))
    modele_cm.fit(X_sin, cm_obs, **{poids_cm: nb_sin}, **(fit_cm or {}))

    equilibrage[nom] = coef_equilibrage(modele_freq, modele_cm, X_tr, y_tr["CHARGE"], expo_tr)
    freq_val = modele_freq.predict(X_val)
    cm_val = modele_cm.predict(X_val) * equilibrage[nom]
    print(f"Coefficient d'équilibrage : {equilibrage[nom]:.3f}")
    evaluer(nom, freq_val * cm_val * expo_val, duree=time.time() - t0)
    return modele_freq, modele_cm

### 5.1 GLM (Poisson × Gamma)

C'est le modèle de référence en tarification : lisible et standard en actuariat. Les variables sont imputées par la médiane, puis standardisées. Une pénalisation Ridge (`alpha`) stabilise l'estimation, car on a ≈ 370 variables pour seulement ≈ 2 300 sinistres côté coût moyen.

In [ ]:
glm_freq = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                         PoissonRegressor(alpha=1e-3, max_iter=500))
glm_cm = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                       GammaRegressor(alpha=1.0, max_iter=500))

glm_freq, glm_cm = evaluer_freq_cm(
    "GLM Poisson × Gamma", glm_freq, glm_cm,
    poids_freq="poissonregressor__sample_weight", poids_cm="gammaregressor__sample_weight",
)

### 5.2 HistGradientBoosting (scikit-learn)

C'est un gradient boosting natif de scikit-learn, avec les pertes `poisson` et `gamma`. Le nombre d'itérations est fixé par *early stopping* sur une fraction interne du train (10 %), ce qui évite d'utiliser le jeu de validation.

In [ ]:
hgb_freq = HistGradientBoostingRegressor(
    loss="poisson", learning_rate=0.05, max_iter=1000, max_leaf_nodes=31,
    min_samples_leaf=200, l2_regularization=1.0,
    early_stopping=True, validation_fraction=0.1, n_iter_no_change=50, random_state=SEED,
)
hgb_cm = HistGradientBoostingRegressor(
    loss="gamma", learning_rate=0.03, max_iter=500, max_leaf_nodes=15,
    min_samples_leaf=30, l2_regularization=1.0,
    early_stopping=True, validation_fraction=0.15, n_iter_no_change=50, random_state=SEED,
)
hgb_freq, hgb_cm = evaluer_freq_cm("HistGradientBoosting Poisson × Gamma", hgb_freq, hgb_cm)
print(f"Itérations retenues — fréquence : {hgb_freq.n_iter_}, coût moyen : {hgb_cm.n_iter_}")

### 5.3 LightGBM (Poisson × Gamma)

Le nombre d'arbres est choisi par *early stopping* sur le jeu de validation, puis réutilisé pour le modèle final.

In [ ]:
lgb_freq = lgb.LGBMRegressor(
    objective="poisson", n_estimators=3000, learning_rate=0.03, num_leaves=63,
    min_child_samples=200, subsample=0.9, subsample_freq=1, colsample_bytree=0.8,
    reg_lambda=1.0, random_state=SEED, verbose=-1,
)
lgb_cm = lgb.LGBMRegressor(
    objective="gamma", n_estimators=2000, learning_rate=0.02, num_leaves=15,
    min_child_samples=30, subsample=0.9, subsample_freq=1, colsample_bytree=0.8,
    reg_lambda=1.0, random_state=SEED, verbose=-1,
)
lgb_freq, lgb_cm = evaluer_freq_cm(
    "LightGBM Poisson × Gamma", lgb_freq, lgb_cm,
    fit_freq={"eval_set": [(X_val, y_val["FREQ"])], "eval_sample_weight": [expo_val],
              "callbacks": [lgb.early_stopping(100, verbose=False)]},
    fit_cm={"eval_set": [(X_val[sev_val.values], y_val.loc[sev_val, "CM"])],
            "eval_sample_weight": [poids_sev(y_val[sev_val])],
            "callbacks": [lgb.early_stopping(100, verbose=False)]},
)
print(f"Arbres retenus — fréquence : {lgb_freq.best_iteration_}, coût moyen : {lgb_cm.best_iteration_}")

### 5.4 XGBoost (Tweedie, prime pure directe)

Cette variante modélise directement la **prime pure** (`CHARGE / exposition`), pondérée par l'exposition. La loi **Tweedie** (1 < p < 2) convient bien à cette cible : elle combine une masse en zéro (pas de sinistre) et une partie continue positive (le coût). Elle est mieux adaptée qu'une perte quadratique sur `CHARGE`. Les quelques charges négatives sont ramenées à 0, et le même équilibrage global est appliqué.

In [ ]:
pp_tr = (y_tr["CHARGE"].clip(lower=0) / expo_tr).to_numpy()
pp_val = (y_val["CHARGE"].clip(lower=0) / expo_val).to_numpy()

xgb_pp = xgb.XGBRegressor(
    objective="reg:tweedie", tweedie_variance_power=1.5,
    n_estimators=3000, learning_rate=0.03, max_depth=6, min_child_weight=50,
    subsample=0.9, colsample_bytree=0.8, reg_lambda=1.0,
    tree_method="hist", early_stopping_rounds=100, random_state=SEED,
)
t0 = time.time()
xgb_pp.fit(X_tr, pp_tr, sample_weight=expo_tr,
           eval_set=[(X_val, pp_val)], sample_weight_eval_set=[expo_val], verbose=False)
# Même équilibrage : la charge totale prédite sur le train doit égaler la charge observée
equilibrage["XGBoost"] = np.sum(expo_tr * pp_tr) / np.sum(expo_tr * xgb_pp.predict(X_tr))
print(f"Coefficient d'équilibrage : {equilibrage['XGBoost']:.3f}")
evaluer("XGBoost Tweedie (prime pure)", xgb_pp.predict(X_val) * equilibrage["XGBoost"] * expo_val,
        duree=time.time() - t0)
print(f"Arbres retenus : {xgb_pp.best_iteration + 1}")

## 6. Comparaison des modèles

In [ ]:
tableau = pd.DataFrame(resultats).T.sort_values("RMSE validation")
reference = tableau.loc["Référence : prime pure moyenne", "RMSE validation"]
tableau["Gain vs référence (%)"] = 100 * (reference - tableau["RMSE validation"]) / reference
tableau

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
couleurs = ["#999999" if n.startswith("Référence") else "#4C72B0" for n in tableau.index][::-1]
noms = tableau.index[::-1]

axes[0].barh(noms, tableau["RMSE validation"][::-1], color=couleurs)
axes[0].set_xlim(tableau["RMSE validation"].min() * 0.995, tableau["RMSE validation"].max() * 1.002)
axes[0].set_title("RMSE validation (plus bas = meilleur)")

axes[1].barh(noms, tableau["Gini normalisé"][::-1], color=couleurs)
axes[1].set_xlim(0, 1)
axes[1].set_title("Gini normalisé (plus haut = meilleure segmentation)")

plt.tight_layout()
plt.show()

In [ ]:
# Variables les plus utilisées par le modèle de fréquence LightGBM
importance = (pd.Series(lgb_freq.booster_.feature_importance("gain"), index=X_tr.columns)
                .sort_values(ascending=False).head(20))
libelles = pd.read_excel(DATA / "dictionnaire_variables.xlsx").set_index("Variable communiquée")["Libellé communiqué"]

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh([f"{v} – {libelles.get(v, '')}"[:45] for v in importance.index[::-1]], importance[::-1], color="#4C72B0")
ax.set_title("Top 20 des variables – fréquence (LightGBM, gain)")
plt.tight_layout()
plt.show()

**Lecture des résultats**

- **Le RMSE départage très peu les modèles**, car il est écrasé par quelques sinistres graves. Même un tarif constant (la référence) obtient un RMSE proche des meilleurs modèles. Des écarts de quelques dizaines d'unités ne sont pas significatifs.
- **Le Gini révèle le vrai apport des modèles** : ils classent les contrats beaucoup mieux que la référence. C'est ce qui permet de tarifer plus cher les risques élevés et moins cher les autres.
- Le **ratio charge prédite / observée** vérifie que le niveau global de sinistralité est respecté.
- Le modèle qui a le plus faible RMSE de validation, la métrique du challenge, est réentraîné sur l'ensemble du train pour produire la soumission.

## 7. Modèle final et fichier de soumission

Le meilleur modèle est réentraîné sur **100 % du train**. Les modèles à early stopping réutilisent le nombre d'itérations trouvé à l'étape 5. Le fichier produit respecte le format attendu : `ID, FREQ, CM, ANNEE_ASSURANCE, CHARGE`.

In [ ]:
meilleur = tableau.drop(index="Référence : prime pure moyenne")["RMSE validation"].idxmin()
print(f"Modèle retenu : {meilleur}")

sev_all = ((y["FREQ"] > 0) & (y["CM"] > 0)).values
X_sev, y_sev, w_sev = X_all[sev_all], y.loc[sev_all, "CM"], poids_sev(y[sev_all])

# Même modèle, mêmes hyperparamètres, avec le nombre d'itérations trouvé en validation
if meilleur.startswith("GLM"):
    f = clone(glm_freq).fit(X_all, y["FREQ"], poissonregressor__sample_weight=exposition)
    c = clone(glm_cm).fit(X_sev, y_sev, gammaregressor__sample_weight=w_sev)
elif meilleur.startswith("HistGradientBoosting"):
    f = clone(hgb_freq).set_params(max_iter=hgb_freq.n_iter_, early_stopping=False)
    c = clone(hgb_cm).set_params(max_iter=hgb_cm.n_iter_, early_stopping=False)
    f.fit(X_all, y["FREQ"], sample_weight=exposition)
    c.fit(X_sev, y_sev, sample_weight=w_sev)
else:  # LightGBM, également utilisé pour la fréquence si XGBoost est retenu
    f = clone(lgb_freq).set_params(n_estimators=lgb_freq.best_iteration_)
    c = clone(lgb_cm).set_params(n_estimators=lgb_cm.best_iteration_)
    f.fit(X_all, y["FREQ"], sample_weight=exposition)
    c.fit(X_sev, y_sev, sample_weight=w_sev)

coef = coef_equilibrage(f, c, X_all, y["CHARGE"], exposition)
print(f"Coefficient d'équilibrage (train complet) : {coef:.3f}")
freq_test, cm_test = f.predict(X_test), c.predict(X_test) * coef
charge_test = freq_test * cm_test * exposition_test

if meilleur.startswith("XGBoost"):
    # XGBoost prédit directement la prime pure : la charge vient de lui,
    # et le coût moyen est déduit pour rester cohérent avec CHARGE = FREQ × CM × exposition
    m = clone(xgb_pp).set_params(n_estimators=xgb_pp.best_iteration + 1, early_stopping_rounds=None)
    pp_all = (y["CHARGE"].clip(lower=0) / exposition).to_numpy()
    m.fit(X_all, pp_all, sample_weight=exposition)
    coef_xgb = np.sum(exposition * pp_all) / np.sum(exposition * m.predict(X_all))
    charge_test = m.predict(X_test) * coef_xgb * exposition_test
    cm_test = charge_test / (freq_test * exposition_test)

soumission = pd.DataFrame({
    "ID": ids_test,
    "FREQ": freq_test,
    "CM": cm_test,
    "ANNEE_ASSURANCE": exposition_test,
    "CHARGE": charge_test,
})
chemin = SORTIES / "submission.csv"
soumission.to_csv(chemin, index=False)

print(f"Fichier écrit : {chemin.relative_to(RACINE)} ({len(soumission):,} lignes)")
print(f"Prime pure moyenne prédite sur le test : {charge_test.sum() / exposition_test.sum():,.1f} "
      f"(train : {y['CHARGE'].sum() / exposition.sum():,.1f})")
soumission.head()

## Conclusion

- L'approche **fréquence × coût moyen**, avec des lois adaptées (Poisson, Gamma, Tweedie) et des pondérations actuarielles (exposition, nombre de sinistres), structure tous les modèles.
- Les modèles de **gradient boosting** captent des interactions que le GLM ne voit pas. Le GLM reste la référence lisible pour expliquer un tarif.
- **Pistes d'amélioration** :
  - écrêter les sinistres graves et mutualiser leur surcoût ;
  - faire une validation croisée par année d'exercice (`AN_EXERC`) ;
  - encoder les catégories par la cible (*target encoding*) ;
  - optimiser les hyperparamètres avec Optuna.